# Cookie Factory Dispatch Plot

This notebook evaluates Cookie Factory SAC and TD3 policies on the same 24-hour price and wind window as the Pyomo model, then compares their power balance (wind, grid, HTHP, and electricity price), grid-power dispatch, and objective value.

In [7]:
import os
import random
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"  # must be before pyplot import

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "cookie_model").exists():
            return candidate
    raise FileNotFoundError("Could not find the THESIS repository root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_FILE = ROOT / "cookie_model" / "WKA_Pe_merged.csv"
# Choose one or both algorithms: ("SAC",), ("TD3",), or ("SAC", "TD3").
RL_ALGORITHMS = ("TD3",)
RL_MODEL_PATHS = {
    "SAC": ROOT / "src" / "cookie_factory" / "cookie_SAC_terminal",
    "TD3": ROOT / "src" / "cookie_factory" / "cookie_TD3_terminal",
}
OUTPUT_FIGURE = ROOT / "notebooks" / "cookie_factory_dispatch_plot.png"
OUTPUT_ACTION_FIGURE = ROOT / "notebooks" / "cookie_factory_action_comparison.png"

WINDOW_START = 0
WINDOW_HOURS = 24
RL_SEED = 42
OPTIMIZATION_STARTS = 1  # Increase for a multistart NLP comparison.
OPTIMIZATION_SEED = 42

plt.rcParams.update(
    {"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False}
)
ROOT

PosixPath('/home/ashish/Documents/THESIS')

In [8]:
from src.cookie_factory.train import load_cookie_data

prices_full, wind_full, price_scale, wind_scale = load_cookie_data(
    str(DATA_FILE), price_col="Pe", wind_col="WKA"
)
window_end = WINDOW_START + WINDOW_HOURS
if window_end > len(prices_full):
    raise ValueError(f"Requested window {WINDOW_START}:{window_end}, but data has {len(prices_full)} rows.")

prices = prices_full[WINDOW_START:window_end]
wind = wind_full[WINDOW_START:window_end]
input_df = pd.DataFrame({"hour": np.arange(WINDOW_HOURS), "price_eur_per_mwh": prices, "wind_power_kw": wind})
print(f"Using hours {WINDOW_START}--{window_end - 1} from {DATA_FILE.relative_to(ROOT)}")
input_df.head()

Loaded 17413 timesteps from /home/ashish/Documents/THESIS/cookie_model/WKA_Pe_merged.csv
price_scale (95th pct): 51.3580
wind_scale  (95th pct): 1780.6771
Using hours 0--23 from cookie_model/WKA_Pe_merged.csv


,hour,price_eur_per_mwh,wind_power_kw
0,0,22.34,429.172328
1,1,17.93,155.994241
2,2,15.17,572.830634
3,3,16.38,1175.252859
4,4,17.38,1420.583334


In [9]:
def build_dispatch_frame(method, price, wind, trajectory):
    trajectory = trajectory.reset_index(drop=True)
    beta1 = trajectory["action_beta1"] if "action_beta1" in trajectory else trajectory["beta1"]
    compressor_speed = trajectory["action_R"] if "action_R" in trajectory else trajectory["R"]
    mass_flow = trajectory["action_m_I"] if "action_m_I" in trajectory else trajectory["m_I"]
    grid_power = trajectory["P_grid"].to_numpy(dtype=float)
    hthp_power = (
        trajectory["P_HP"].to_numpy(dtype=float)
        if "P_HP" in trajectory
        else grid_power + np.asarray(wind, dtype=float)
    )
    return pd.DataFrame({
        "hour": np.arange(len(trajectory)),
        "price_eur_per_mwh": np.asarray(price, dtype=float),
        "wind_power_kw": np.asarray(wind, dtype=float),
        "grid_power_kw": grid_power,
        "hthp_power_kw": hthp_power,
        "storage_temperature_c": trajectory["T_s"].to_numpy(dtype=float),
        "charge_flow_kw": trajectory["Q_ch"].to_numpy(dtype=float),
        "discharge_flow_kw": trajectory["Q_dch"].to_numpy(dtype=float),
        "beta1": beta1.to_numpy(dtype=float),
        "compressor_speed": compressor_speed.to_numpy(dtype=float),
        "mass_flow": mass_flow.to_numpy(dtype=float),
        "cost_eur": (np.asarray(price, dtype=float) / 1000.0) * grid_power,
        "method": method,
    })


def run_rl_dispatch(algorithm, price, wind):
    from src.cookie_factory.evaluate import evaluate_model

    algorithm = algorithm.upper()
    if algorithm not in RL_MODEL_PATHS:
        raise ValueError(f"Unsupported algorithm: {algorithm}")
    model_path = RL_MODEL_PATHS[algorithm]
    vecnormalize_path = model_path.with_name(model_path.name + "_vecnormalize.pkl")
    if not model_path.with_suffix(".zip").exists():
        raise FileNotFoundError(f"Missing saved {algorithm} model: {model_path.with_suffix('.zip')}")

    _, trajectory = evaluate_model(
        model_path=str(model_path),
        g_grid_full=np.asarray(price, dtype=float),
        P_WT_full=np.asarray(wind, dtype=float),
        price_scale=price_scale,
        wind_scale=wind_scale,
        num_episodes=1,
        window_len=len(price),
        forecast_h=len(price),
        start_hour=0,
        deterministic=True,
        seed=RL_SEED,
        vecnormalize_path=str(vecnormalize_path) if vecnormalize_path.exists() else None,
        algorithm=algorithm,
    )
    return build_dispatch_frame(f"RL {algorithm}", price, wind, trajectory)


def run_optimization_dispatch(price, wind):
    from cookie_model.cookie_optimization import run_multistart

    random.seed(OPTIMIZATION_SEED)
    np.random.seed(OPTIMIZATION_SEED)
    starts, result = run_multistart(
        price, wind, starts=OPTIMIZATION_STARTS, verbose=False
    )
    display(starts)
    if result is None:
        raise RuntimeError("No IPOPT start reached an optimal solution.")
    return build_dispatch_frame("optimization baseline", price, wind, result["trajectory"]), result


rl_dispatches = {
    algorithm.upper(): run_rl_dispatch(algorithm, prices, wind)
    for algorithm in RL_ALGORITHMS
}
# Backwards-compatible name for single-policy code below.
rl_dispatch_df = rl_dispatches[RL_ALGORITHMS[0].upper()]
import time
st=time.time()
optimization_dispatch_df, optimization_result = run_optimization_dispatch(prices, wind)
print(time.time()-st)
print(f"Prepared {', '.join(f'RL {algorithm}' for algorithm in rl_dispatches)} and optimization-baseline dispatches.")

Loaded model from /home/ashish/Documents/THESIS/src/cookie_factory/cookie_TD3_terminal.zip
Loading VecNormalize stats from /home/ashish/Documents/THESIS/src/cookie_factory/cookie_TD3_terminal_vecnormalize.pkl
Episode 0: reward=-1949.14, cost=1244.27, violations=924.44, steps=24


,start,status,cost,solver_time_s,error
0,1,success,1130.699303,16.365452,


16.3853542804718
Prepared RL TD3 and optimization-baseline dispatches.


In [10]:
def summarize_dispatch(df, optimization_cost=None):
    grid_power = df["grid_power_kw"]
    summary = {
        "method": df["method"].iloc[0],
        "objective_eur": float(df["cost_eur"].sum()),
        "charge_energy_kwh": float(df["charge_flow_kw"].sum()),
        "discharge_energy_kwh": float(df["discharge_flow_kw"].sum()),
        "min_storage_temperature_c": float(df["storage_temperature_c"].min()),
        "max_storage_temperature_c": float(df["storage_temperature_c"].max()),
        "terminal_temperature_error_c": float(df["storage_temperature_c"].iloc[-1] - 250.0),
        "grid_bound_violation_kw": float(np.maximum(-grid_power, 0.0).sum() + np.maximum(grid_power - 5000.0, 0.0).sum()),
    }
    if optimization_cost is not None:
        summary["solver_objective_eur"] = float(optimization_cost)
    return summary

comparison_df = pd.DataFrame(
    [summarize_dispatch(df) for df in rl_dispatches.values()]
    + [summarize_dispatch(optimization_dispatch_df, optimization_result["cost"])]
)
comparison_df["objective_gap_to_optimization_eur"] = comparison_df["objective_eur"] - float(optimization_result["cost"])
display(comparison_df)
rl_dispatch_df.head()

,method,objective_eur,charge_energy_kwh,discharge_energy_kwh,min_storage_temperature_c,max_storage_temperature_c,terminal_temperature_error_c,grid_bound_violation_kw,solver_objective_eur,objective_gap_to_optimization_eur
0,RL TD3,1244.271358,3894.832102,5135.875228,227.494387,257.558047,-7.264643,917.170903,NaN,113.572055
1,optimization baseline,1130.699303,12683.680904,12683.680904,238.869146,312.575171,0.000000,0.000000,1130.699303,0.000000


,hour,price_eur_per_mwh,wind_power_kw,grid_power_kw,hthp_power_kw,storage_temperature_c,charge_flow_kw,discharge_flow_kw,beta1,compressor_speed,mass_flow,cost_eur,method
0,0,22.34,429.172328,2630.435418,3059.607751,250.000000,0.000000,0.0,1.000000,1.305913,6.606112,58.763927,RL TD3
1,1,17.93,155.994241,2953.893986,3109.888233,250.449720,76.827168,0.0,0.953819,1.311478,7.054718,52.963319,RL TD3
2,2,15.17,572.830634,2646.515622,3219.346249,251.725863,218.007680,0.0,0.865199,1.324822,7.588162,40.147642,RL TD3
3,3,16.38,1175.252859,2187.934332,3363.187140,254.031969,393.959919,0.0,0.754675,1.342945,7.964849,35.838364,RL TD3
4,4,17.38,1420.583334,1872.753402,3293.336776,255.847454,310.145295,0.0,0.802938,1.335094,7.472815,32.548454,RL TD3


In [11]:
def step_values(hours, values):
    return np.r_[hours, hours[-1] + 1], np.r_[values, values[-1]]


def plot_cookie_dispatch(rl_df, math_df, output_path=None):
    hours = rl_df["hour"].to_numpy(dtype=int)
    x, price = step_values(hours, rl_df["price_eur_per_mwh"].to_numpy(dtype=float))
    _, wind_available = step_values(hours, rl_df["wind_power_kw"].to_numpy(dtype=float))
    _, rl_grid = step_values(hours, rl_df["grid_power_kw"].to_numpy(dtype=float))
    _, rl_hthp = step_values(hours, rl_df["hthp_power_kw"].to_numpy(dtype=float))
    _, math_grid = step_values(hours, math_df["grid_power_kw"].to_numpy(dtype=float))

    # fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True, gridspec_kw={"hspace": 0.12})
    # ax_balance, ax_grid, ax_flow, ax_storage = axes
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"hspace": 0.12})
    ax_balance, ax_grid = axes

    balance_wind_fill = ax_balance.fill_between(x, 0.0, wind_available, step="post", color="#78a984", alpha=0.65, label=r"Wind power $W_t$")
    balance_grid_fill = ax_balance.fill_between(x, wind_available, rl_hthp, step="post", color="#7b83d1", alpha=0.65, label=r"Grid power $P_t^{\mathrm{S}}$")
    balance_hthp_line = ax_balance.step(x, rl_hthp, where="post", color="#222222", linestyle="--", linewidth=2.0, label=r"HTHP demand")[0]
    ax_balance.set_ylabel("Power [kW]", fontweight="bold")
    ax_balance.grid(True, alpha=0.28)
    ax_balance_price = ax_balance.twinx()
    balance_price_line = ax_balance_price.plot(x, price, color="#d62728", linewidth=2.0, label=r"Electricity price $S_t$")[0]
    ax_balance_price.set_ylabel("Electricity price [EUR/MWh]", fontweight="bold")

    rl_grid_fill = ax_grid.fill_between(x, 0.0, rl_grid, step="post", color="#7b83d1", alpha=0.55, label=r"RL grid power $P_t^{\mathrm{S}}$")
    ax_grid.step(x, rl_grid, where="post", color="#4f56a6", linewidth=1.2)
    math_grid_line = ax_grid.step(x, math_grid, where="post", color="#2455a4", linestyle="--", linewidth=2.2, label=r"Baseline grid power $P_t^{\mathrm{S}}$")[0]
    ax_grid.set_ylabel("Grid power [kW]", fontweight="bold")
    ax_grid.grid(True, alpha=0.28)
    # Net-thermal-flow panel omitted from the thesis dispatch figure.
    # for frame, color, style, label in [(rl_df, "#0b5d1e", "-", r"RL thermal flow ($Q_{ch} - Q_{dch}$)"), (math_df, "#2455a4", "--", r"Math thermal flow ($Q_{ch} - Q_{dch}$)")]:
    #     net_flow = frame["charge_flow_kw"].to_numpy(dtype=float) - frame["discharge_flow_kw"].to_numpy(dtype=float)
    #     xf, flow = step_values(hours, net_flow)
    #     ax_flow.step(xf, flow, where="post", color=color, linestyle=style, linewidth=2.0, label=label)
    # ax_flow.axhline(0.0, color="#222222", linewidth=1.0)
    # ax_flow.set_ylabel(r"Net thermal flow, $Q_{ch} - Q_{dch}$ [kW]", fontweight="bold")
    # ax_flow.grid(True, alpha=0.28)

    # Storage-temperature panel omitted from the thesis dispatch figure.
    # rl_storage_line = ax_storage.plot(hours, rl_df["storage_temperature_c"], color="#222222", linewidth=2.1, label=r"RL storage temperature ($T_s$)")[0]
    # math_storage_line = ax_storage.plot(hours, math_df["storage_temperature_c"], color="#5f5f5f", linestyle="--", linewidth=2.1, label=r"Math storage temperature ($T_s$)")[0]
    # ax_storage.axhline(250.0, color="#999999", linewidth=1.0, linestyle=":", label="Cyclic target")
    # ax_storage.set_xlabel("Time [h]")
    # ax_storage.set_ylabel("Storage temperature [°C]", fontweight="bold")
    # ax_storage.set_xticks(np.arange(hours[0], hours[-1] + 2))
    # ax_storage.grid(True, alpha=0.28)
    ax_grid.set_xlabel("Time [h]")
    ax_grid.set_xticks(np.arange(hours[0], hours[-1] + 2))
    ax_grid.set_xlim(hours[0], hours[-1] + 1)

    rl_method = rl_df["method"].iloc[0]
    fig.suptitle(f"Cookie Factory dispatch: {rl_method} vs optimization baseline | hours {WINDOW_START}-{window_end - 1}", fontsize=14, y=0.99)
    fig.text(0.5, 0.925, f"{rl_method} objective: {rl_df.cost_eur.sum():.2f} EUR  |  Baseline objective: {optimization_result['cost']:.2f} EUR", ha="center", fontsize=10)
    # flow_handles, flow_labels = ax_flow.get_legend_handles_labels()
    fig.legend(
        [balance_wind_fill, balance_grid_fill, balance_hthp_line, balance_price_line, math_grid_line],
        [r"Wind power $W_t$", r"RL grid power $P_t^{\mathrm{S}}$", r"HTHP demand", r"Electricity price $S_t$", r"Baseline grid power $P_t^{\mathrm{S}}$"],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.875),
        ncol=5,
        fontsize=9,
        frameon=True,
    )
    fig.subplots_adjust(top=0.80)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")
    return fig


for algorithm, algorithm_dispatch_df in rl_dispatches.items():
    output_path = OUTPUT_FIGURE if algorithm == "SAC" else OUTPUT_FIGURE.with_name(
        f"{OUTPUT_FIGURE.stem}_{algorithm.lower()}{OUTPUT_FIGURE.suffix}"
    )
    plot_cookie_dispatch(algorithm_dispatch_df, optimization_dispatch_df, output_path)

Saved figure to notebooks/cookie_factory_dispatch_plot_td3.png


In [12]:
def plot_action_comparison(rl_df, math_df, output_path=None):
    hours = rl_df["hour"].to_numpy(dtype=int)
    action_specs = [
        ("beta1", r"Bypass mode, $\beta_1$ [-]", [0.0, 1.0]),
        ("mass_flow", r"Mass flow, $m_I$ [kg/s]", None),
    ]
    fig, axes = plt.subplots(2, 1, figsize=(14, 5.5), sharex=True, gridspec_kw={"hspace": 0.12})

    for axis, (column, ylabel, ticks) in zip(axes, action_specs):
        x, rl_values = step_values(hours, rl_df[column].to_numpy(dtype=float))
        _, math_values = step_values(hours, math_df[column].to_numpy(dtype=float))
        axis.step(x, rl_values, where="post", color="#0b5d1e", linewidth=2.1, label=r"RL action")
        axis.step(x, math_values, where="post", color="#2455a4", linestyle="--", linewidth=2.1, label=r"Math decision")
        axis.set_ylabel(ylabel, fontweight="bold")
        axis.grid(True, alpha=0.28)
        if ticks is not None:
            axis.set_yticks(ticks)
            axis.set_ylim(-0.08, 1.08)

    axes[-1].set_xlabel("Time [h]")
    axes[-1].set_xticks(np.arange(hours[0], hours[-1] + 2))
    rl_method = rl_df["method"].iloc[0]
    fig.suptitle(f"Cookie Factory action comparison: {rl_method} vs optimization baseline | hours {WINDOW_START}-{window_end - 1}", fontsize=15, y=0.98)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.935), ncol=2, frameon=True)
    fig.subplots_adjust(top=0.86)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to {output_path.relative_to(ROOT)}")
    return fig


for algorithm, algorithm_dispatch_df in rl_dispatches.items():
    output_path = OUTPUT_ACTION_FIGURE if algorithm == "SAC" else OUTPUT_ACTION_FIGURE.with_name(
        f"{OUTPUT_ACTION_FIGURE.stem}_{algorithm.lower()}{OUTPUT_ACTION_FIGURE.suffix}"
    )
    plot_action_comparison(algorithm_dispatch_df, optimization_dispatch_df, output_path)

Saved figure to notebooks/cookie_factory_action_comparison_td3.png


The default window and dataset match the Cookie Factory training and optimization setup. Set `RL_ALGORITHMS` to `("SAC",)`, `("TD3",)`, or `("SAC", "TD3")` to select policies; when both are selected, the notebook produces one dispatch and action-comparison plot per algorithm. Set `WINDOW_START` to compare another 24-hour segment, and increase `OPTIMIZATION_STARTS` if a stronger multistart NLP baseline is required. Running the optimization requires a working IPOPT installation.